<a href="https://colab.research.google.com/github/Suskan86/Artificial-Intelligence/blob/main/Model_Deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Train a simple model (using Iris dataset as example)
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import joblib

X, y = load_iris(return_X_y=True)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

joblib.dump(model, "model.pkl")
print("Model saved as model.pkl")

Model saved as model.pkl


In [ ]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

app = FastAPI(title="Model Inference API")
model = joblib.load("model.pkl")

class InputData(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

@app.get("/")
def home():
    return {"message": "API is running"}

@app.post("/predict")
def predict(data: InputData):
    features = np.array([[data.sepal_length, data.sepal_width,
                           data.petal_length, data.petal_width]])
    prediction = model.predict(features)
    return {"prediction": int(prediction[0])}

Overwriting app.py


In [ ]:
%%writefile requirements.txt
fastapi
uvicorn
scikit-learn
joblib
numpy

Overwriting requirements.txt


In [ ]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py model.pkl ./

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

Overwriting Dockerfile


In [ ]:
!pip install fastapi uvicorn pyngrok -q

In [ ]:
import subprocess
import time

# Kill any previous instance
!pkill -f uvicorn

# Start server as a background process (avoids nest_asyncio conflict)
server_process = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
)

time.sleep(3)  # give it a moment to start
print("Server started with PID:", server_process.pid)

Server started with PID: 8061


In [ ]:
import requests

response = requests.post(
    "http://localhost:8000/predict",
    json={
        "sepal_length": 5.1,
        "sepal_width": 3.5,
        "petal_length": 1.4,
        "petal_width": 0.2
    }
)
print(response.status_code)
print(response.json())

200
{'prediction': 0}
